# Real-Time Interaction with Streaming

Hello everyone. So far, our API calls have followed a standard pattern: we send a request, wait for the *entire* response to be generated, and then receive it all at once. This is fine for quick tasks, but for longer generations, it can lead to a poor user experience with a lot of waiting.

This is where **streaming** comes in. Instead of waiting for the full response, streaming lets us receive the output **a few tokens at a time, as they're being generated.** This is how applications like ChatGPT feel so fast and responsive.

Let's see how to implement this simple but powerful feature.

## Setup

To appreciate the benefit of streaming, we need a task that isn't instantaneous. We'll ask the model to generate a detailed explanation of a programming concept. This will take a few seconds, making the difference between blocking and streaming very clear.

In [3]:
import time
import litellm
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS = 200

long_prompt = [
    {
        "role": "user",
        "content": dedent("""
            You are a senior Python developer tutoring a junior colleague.
            Explain the difference between concurrency and parallelism in Python.
            Keep it concise.
        """)
    }
]

## The Blocking Request

First, let's make a standard API call. When you run the cell below, pay close attention to the pause between when you execute it and when the full text appears. This waiting period is what we want to eliminate.

In [5]:
print("--- Making a standart blocking call ----")
start_time = time.perf_counter()

response_blocking = litellm.completion(
    model=MODEL_NAME, 
    max_tokens=MAX_TOKENS, 
    messages=long_prompt
)

content = response_blocking.choices[0].message.content
end_time = time.perf_counter()
print(f"Blocking call took {end_time - start_time:.2f} seconds")
print(f"Content: {content}")

--- Making a standart blocking call ----
Blocking call took 1.91 seconds
Content: Sure! 

**Concurrency** and **parallelism** are related concepts but differ in execution.

- **Concurrency** refers to the ability of a program to handle multiple tasks at once by managing their execution, typically through a single-threaded process. In Python, this can be achieved using `asyncio`, where tasks are interleaved, allowing for I/O-bound operations to continue while waiting for others to complete. Think of it as planning a schedule where tasks overlap.

- **Parallelism**, on the other hand, refers to the actual simultaneous execution of multiple processes or threads, often on multiple CPU cores. This can be achieved in Python using the `multiprocessing` module, which allows for true parallel execution of CPU-bound tasks. It's like having multiple workers who can work on different parts of a problem at the same time.

In summary:
- **Concurrency** = interleaving of tasks (often using async prog

## The Streaming Request

Now, let's make the exact same request, but this time we'll add one crucial parameter: `stream=True`.

When we do this, the function no longer returns a complete response object. Instead, it returns a **generator**. We can then loop through this generator to get each "chunk" of the response as it's created. This allows us to print the text to the screen almost instantly.

In [10]:
print("--- Making a streaming call ----")
start_time = time.perf_counter()
full_response_streaming = "" 
response_streaming = litellm.completion(
    model=MODEL_NAME, 
    max_tokens=MAX_TOKENS, 
    messages=long_prompt,
    stream=True
)
for chunk in response_streaming:
    if chunk.choices[0].delta.content:
        chunk_content = chunk.choices[0].delta.content
        full_response_streaming += chunk_content
        print(chunk_content, end="", flush=True)
    
end_time = time.perf_counter()
print(f"\n\nStreaming call took {end_time - start_time:.2f} seconds")
# print(f"Full response: {full_response_streaming}")

--- Making a streaming call ----
Certainly! 

**Concurrency** and **parallelism** are related concepts but serve different purposes, especially in the context of Python.

- **Concurrency** refers to the ability of a program to manage multiple tasks at the same time. This does not necessarily mean that these tasks are executed simultaneously. Instead, they can be interleaved in a single-threaded environment. Python achieves concurrency primarily through the `asyncio` library, which uses asynchronous programming (cooperative multitasking) to allow multiple tasks to run in overlapping periods without blocking each other.

- **Parallelism**, on the other hand, is the actual simultaneous execution of multiple tasks, often leveraging multiple CPU cores. In Python, you can achieve parallelism using the `multiprocessing` module, which spawns separate processes for each task. This is particularly useful for CPU-bound tasks, as it bypasses Python's Global Interpreter Lock (GIL), allowing multipl